# **Random Forest - Regressão**

In [1]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.model_selection import RandomizedSearchCV
import warnings 

warnings.filterwarnings("ignore")

Vou abordar um problema de previsão de preços de aluguel. O objetivo é prever o valor do aluguel de imóveis com base em diversas características como localização, número de quartos etc.

Random Forest é uma técnica versátil de aprendizado de máquina que pode ser utilizada tanto para problemas de regressão quanto para classificação assim como as árvores de decisão.

No caso da regressão, o Random Forest faz a previsão tomando a média das previsões de todas as árvores da floresta. Cada árvore individual faz uma estimativa para o valor contínuo do alvo, e o Random Forest combina essas estimativas, calculando a média para produzir a previsão final. Isso ajuda a suavizar as previsões e a reduzir a variabilidade, resultando em um modelo mais estável e preciso.

In [2]:
base_imov = pd.read_csv("../../Base de Dados/ALUGUEL_MOD12.csv", delimiter=';')

In [3]:
base_imov.describe()

,Valor_Aluguel,Valor_Condominio,Metragem,N_Quartos,N_banheiros,N_Suites,N_Vagas
count,7203.000000,7203.000000,7203.000000,7203.000000,7203.000000,7203.000000,7203.00000
mean,2966.596140,811.538109,88.506178,2.300153,2.095932,1.016660,1.44176
std,2948.720385,796.564846,61.567505,0.826615,0.983812,0.874204,0.86993
min,480.000000,0.000000,30.000000,1.000000,1.000000,0.000000,0.00000
25%,1350.000000,395.000000,52.000000,2.000000,2.000000,1.000000,1.00000
50%,2000.000000,592.000000,67.000000,2.000000,2.000000,1.000000,1.00000
75%,3200.000000,980.000000,100.000000,3.000000,2.000000,1.000000,2.00000
max,25000.000000,9500.000000,880.000000,10.000000,8.000000,5.000000,9.00000


In [4]:
X = base_imov.drop('Valor_Aluguel', axis=1)
y = base_imov['Valor_Aluguel']

In [5]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

In [6]:
# Notem que usamos o random forest regressor
rf_model_reg = RandomForestRegressor(random_state=42)

In [7]:
rf_model_reg.fit(X_train, y_train)

RandomForestRegressor(random_state=42)

In [8]:
y_pred_reg = rf_model_reg.predict(X_test)

In [9]:
mse = mean_squared_error(y_test, y_pred_reg)
mae = mean_absolute_error(y_test, y_pred_reg)
r2 = r2_score(y_test, y_pred_reg)

In [10]:
print(f"RMSE: {mse ** 0.5}")
print(f"MAE: {mae}")
print(f"R²: {r2}")

RMSE: 1872.2883535577132
MAE: 968.6055994375845
R²: 0.6434247719501688


O RMSE é a raiz quadrada da média dos erros quadráticos.
Um RMSE de 1872.29 significa que, em média, a diferença entre os valores previstos e os valores reais é de aproximadamente 1872.29 unidades (provavelmente na mesma unidade que os preços de aluguel).

O MAE é a média dos erros absolutos.
Um MAE de 968.61 significa que, em média, a diferença entre os valores previstos e os valores reais é de aproximadamente 968.61 unidades.

O R² mede a proporção da variância nos dados de resposta que é explicada pelo modelo. Um R² de 0.6434 indica que aproximadamente 64.34% da variância nos preços de aluguel pode ser explicada pelas variáveis independentes (features) no modelo.

## **Melhorando os hyperparametros:**

In [11]:
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_features': ['sqrt', 'log2'],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

In [12]:
rf_model_reg_best = RandomForestRegressor(random_state=42)

In [13]:
# Definindo o RandomizedSearchCV para regressão:
random_search_reg = RandomizedSearchCV(estimator=rf_model_reg_best, param_distributions=param_grid,
                                   n_iter=50, cv=5, n_jobs=-1, verbose=2, random_state=42, scoring='neg_mean_squared_error')

In [14]:
random_search_reg.fit(X_train, y_train)

Fitting 5 folds for each of 50 candidates, totalling 250 fits


[CV] END max_depth=30, max_features=log2, min_samples_leaf=2, min_samples_split=5, n_estimators=50; total time=   0.3s
[CV] END max_depth=30, max_features=log2, min_samples_leaf=2, min_samples_split=5, n_estimators=50; total time=   0.3s
[CV] END max_depth=20, max_features=log2, min_samples_leaf=1, min_samples_split=5, n_estimators=50; total time=   0.3s
[CV] END max_depth=20, max_features=log2, min_samples_leaf=1, min_samples_split=5, n_estimators=50; total time=   0.4s
[CV] END max_depth=20, max_features=log2, min_samples_leaf=1, min_samples_split=5, n_estimators=50; total time=   0.3s
[CV] END max_depth=30, max_features=log2, min_samples_leaf=4, min_samples_split=10, n_estimators=50; total time=   0.3s
[CV] END max_depth=30, max_features=sqrt, min_samples_leaf=2, min_samples_split=10, n_estimators=50; total time=   0.3s
[CV] END max_depth=30, max_features=sqrt, min_samples_leaf=2, min_samples_split=10, n_estimators=50; total time=   0.3s
[CV] END max_depth=30, max_features=sqrt, min

RandomizedSearchCV(cv=5, estimator=RandomForestRegressor(random_state=42),
                   n_iter=50, n_jobs=-1,
                   param_distributions={'max_depth': [None, 10, 20, 30],
                                        'max_features': ['sqrt', 'log2'],
                                        'min_samples_leaf': [1, 2, 4],
                                        'min_samples_split': [2, 5, 10],
                                        'n_estimators': [50, 100, 200]},
                   random_state=42, scoring='neg_mean_squared_error',
                   verbose=2)

In [15]:
best_params_reg = random_search_reg.best_params_
print(f"Melhores Hiperparâmetros: {best_params_reg}")

Melhores Hiperparâmetros: {'n_estimators': 100, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 'sqrt', 'max_depth': 10}


In [16]:
best_rf_model_rg = random_search_reg.best_estimator_
best_rf_model_rg.fit(X_train, y_train)

RandomForestRegressor(max_depth=10, max_features='sqrt', random_state=42)

In [17]:
y_pred_reg2 = best_rf_model_rg.predict(X_test)

In [18]:
mse = mean_squared_error(y_test, y_pred_reg2)
mae = mean_absolute_error(y_test, y_pred_reg2)
r2 = r2_score(y_test, y_pred_reg2)

print(f"Melhores hiperparâmetros: {random_search_reg.best_params_}")
print(f"RMSE: {mse ** 0.5}")
print(f"MAE: {mae}")
print(f"R²: {r2}")

Melhores hiperparâmetros: {'n_estimators': 100, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 'sqrt', 'max_depth': 10}
RMSE: 1858.0023460338516
MAE: 975.1161060374673
R²: 0.6488455203339903
